# JobStack Project 1: EDA (Person 4)

**Data:** `data/transformed/jobs_final.csv` (Person 2's output). Currently a 300-row development sample.
**To use the full data, change only `DATA_PATH` in `src/eda/load_data.py`** (or pass a path to `load_jobs`), then *Kernel -> Restart & Run All*.

All reusable code lives in `src/eda/`. Results here are **associations in advertised postings**, not causes or requirements.

Sections: 1. Overview & missingness | 2. Salary coverage | 3. Skill demand | 4. City patterns (Pakistan) | 5. Experience | 6. Tech-only Pakistan subset | 7. Skill co-occurrence | 8. Role-specific skills | 9. Findings log

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether Jupyter starts in the repo root or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.eda.load_data import load_jobs, tech_pakistan, save_fig, DATA_PATH, TECH_CATEGORIES
from src.eda.skills import (plot_skills, pair_counts, pair_table, plot_cooccurrence,
                            skill_lift, SOFT_SKILLS)
from src.eda.experience import exp_level, LEVEL_ORDER
from src.eda.salary import salary_coverage

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

df = load_jobs()
print(f"Loaded {len(df):,} rows from {DATA_PATH.name}")

## 1. Overview & missingness

In [ ]:
display(df["source"].value_counts().to_frame("rows").assign(pct=lambda t: (100 * t.rows / len(df)).round(1)))

miss = (df.drop(columns=["skills", "n_skills"]).isna().mean() * 100).round(1).sort_values(ascending=False)
display(miss.to_frame("pct_missing"))

In [ ]:
# Missingness by source: shows which fields only exist for some sources
cols = ["city", "experience", "employment_type", "company_name", "category", "salary_text"]
display((df.groupby("source")[cols].apply(lambda g: g.isna().mean() * 100)).round(1))

# Date range per source (sources cover different periods: no cross-source trends)
display(df.groupby("source")["posted_at"].agg(["min", "max"]))

## 2. Salary coverage
Salary is **never treated as zero**. We only report how much usable salary data exists.

In [ ]:
cov = salary_coverage(df)
print(f"Numeric salary:        {cov['numeric']} of {cov['n']} ({cov['numeric_pct']}%)")
print(f"Free-text salary only: {cov['text_only']} ({cov['text_only_pct']}%)")
print(f"No salary info:        {cov['none']} ({cov['none_pct']}%)")

## 3. Skill demand
Percentages are of postings in the group, not of total skill mentions.

In [ ]:
print(f"Postings with at least one extracted skill: {(df.n_skills > 0).mean()*100:.1f}%")
display(df.groupby("source")["n_skills"].mean().round(2).to_frame("mean_skills_per_posting"))
plot_skills(df, "Top skills, all sources", "top_skills_all")

In [ ]:
# Per source: sources describe very different job markets, so compare separately
for src, g in df.groupby("source"):
    if len(g) >= 30:      # skip tiny groups (e.g. remotive in the sample)
        plot_skills(g, f"Top skills, {src}", f"top_skills_{src}")
    else:
        print(f"Skipping {src}: only {len(g)} rows")

## 4. City patterns (Pakistan only)
Only `kaggle_pk` is a Pakistan source. It covers all industries, not just tech.

In [ ]:
pk = df[df["source"] == "kaggle_pk"]
city_counts = pk["standardized_city"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=city_counts.values, y=city_counts.index, color="#10b981", ax=ax)
ax.set_title(f"Postings by city, Pakistan source (n={len(pk)})")
ax.set_xlabel("postings"); ax.set_ylabel("")
save_fig("postings_by_city_pk")
plt.show()

display(pk["category"].value_counts().head(10).to_frame("postings"))

## 5. Experience (Pakistan source)
Raw text like `2 Years Job Exp.` is parsed into rough levels.

In [ ]:
pk = pk.assign(exp_level=pk["experience"].apply(exp_level))
print("Unparsed experience values:", pk["exp_level"].isna().sum(), "of", len(pk))
display(pk["exp_level"].value_counts().reindex(LEVEL_ORDER).to_frame("postings"))
display(pk.groupby("exp_level")["n_skills"].agg(["count", "mean"]).reindex(LEVEL_ORDER).round(2))

## 6. Tech-only Pakistan subset
Adjust `TECH_CATEGORIES` in `src/eda/load_data.py` after reviewing the category list in section 4.

In [ ]:
pk_tech = tech_pakistan(df)
print(f"Pakistan tech postings: {len(pk_tech)}")
print(f"With at least one extracted skill: {(pk_tech.n_skills > 0).mean()*100:.1f}%  <- extraction coverage check")
print("\nTech postings with NO extracted skill (examples for Person 2):")
display(pk_tech.loc[pk_tech.n_skills == 0, ["title", "category"]].head(10))

## 7. Skill co-occurrence
How often two skills appear in the same posting. `MIN_PAIR_COUNT` hides rare pairs: **~5 on the 300-row sample, ~30 on the full data.**
Soft skills (Communication, Teamwork) dominate, so a technical-only view is shown too.

In [ ]:
MIN_PAIR_COUNT = 5

counter, n_scope = pair_counts(df)
print(f"Postings with 2+ skills: {n_scope}")
display(pair_table(counter, n_scope, MIN_PAIR_COUNT).head(15))
plot_cooccurrence(df, counter, n_scope, fname="skill_cooccurrence")

In [ ]:
# Technical skills only (soft skills removed)
counter_t, n_t = pair_counts(df, exclude=SOFT_SKILLS)
print(f"Postings with 2+ technical skills: {n_t}")
display(pair_table(counter_t, n_t, MIN_PAIR_COUNT).head(15))
plot_cooccurrence(df, counter_t, n_t, fname="skill_cooccurrence_technical", exclude=SOFT_SKILLS)

## 8. Role-specific skills (lift)
**Lift** = how much more common a skill is in a group than overall (lift 2 = twice as common).
Small groups and rare skills are ignored. **Association only.** On the sample each cell rests on 3-5 postings, so do not quote these yet.

In [ ]:
lift = skill_lift(df, group_col="category", min_group=8, min_skill_count=3)   # try standardized_title on full data
print(f"Groups analysed: {lift['category'].nunique() if len(lift) else 0}")
if len(lift):
    display(lift.sort_values("lift", ascending=False).groupby("category").head(3)
                .sort_values(["category", "lift"], ascending=[True, False]))

## 9. Findings log
Final findings live in `reports/findings/project1_findings.md`. Record each with: metric, sample size, filter, limitation.